# 把 preprocess.py 拆开，一步一步看

这个 notebook 把 [preprocess.py](preprocess.py) 拆成十几小步。**每一格先打印处理前的样子，再打印处理后的样子**，看清楚数据是怎么一点点变的。

用法：从上到下按 `Shift + Enter` 一格一格执行，重点看每次打印出来的表格。

为了跑得快，这里只读前 5 万行。`preprocess.py` 用的是全部 80 万行，步骤完全一样。

> 记住一句话：**pandas 是对一整列操作，不写 for 循环。** 看懂这一点，下面基本都能顺下来。

## 第 0 步：读数据 + 先拼起来

`pd.read_csv` 把 csv 读成一张表（DataFrame）。

接着做三件事，顺序和 `preprocess.py` 一样：

1. `pop("isDefault")` 把答案列单独拿出来（`pop` = 取出来 + 从表里删掉）
2. **把训练集和测试集上下拼起来**。为什么要先拼？因为后面 one-hot 编码时，如果分开做，某个类别只在一边出现，两边的列就对不齐了。拼在一起处理，保证两边列完全一致，清洗规则也一次搞定。
3. `n_train` 记住训练集有多少行，等会要按这个行号切回来

In [12]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

train = pd.read_csv("train.csv", nrows=50000)
test = pd.read_csv("testA.csv", nrows=50000)

print("train 的形状（行数, 列数）:", train.shape)
print("test  的形状（行数, 列数）:", test.shape, "  ← 少一列 isDefault，因为测试集没有答案")
print()

test_ids = test["id"].to_numpy()
y = train.pop("isDefault").to_numpy()
n_train = len(train)

print("答案单独拿出来了 y:", y.shape, "，里面的违约比例是", round(y.mean(), 4))
print("train pop 完还剩", train.shape[1], "列，和 test 一样多了")
print("记住 n_train =", n_train)

train 的形状（行数, 列数）: (50000, 47)
test  的形状（行数, 列数）: (50000, 46)   ← 少一列 isDefault，因为测试集没有答案

答案单独拿出来了 y: (50000,) ，里面的违约比例是 0.2002
train pop 完还剩 46 列，和 test 一样多了
记住 n_train = 50000


In [13]:
full = pd.concat([train, test], ignore_index=True)

print("拼起来之后 full 的形状:", full.shape)
print("  ", n_train, "(train) +", len(test), "(test) =", n_train + len(test), "← 行数对得上")
print()
print("ignore_index=True 的作用是重新排行号，不然两边的行号都是 0 开始，会重复。")

拼起来之后 full 的形状: (100000, 46)
   50000 (train) + 50000 (test) = 100000 ← 行数对得上

ignore_index=True 的作用是重新排行号，不然两边的行号都是 0 开始，会重复。


## 第 1 步：DataFrame 和 Series

- `DataFrame` = 一张表
- `Series` = 表里的一列

`df["列名"]` 取一列（Series），`df[["列名1", "列名2"]]` 取多列（还是 DataFrame）。

In [14]:
print("类型:", type(full))
print()
one_col = full["annualIncome"] # 取出表头为annualIncome的一列的所有数据
print("取一列:", type(one_col), "→ 这是一个 Series")
print(one_col.head(3))
print()
two_cols = full[["annualIncome", "dti"]] # 取出两列数据就是DataFrame了
print("取多列:", type(two_cols), "→ 还是 DataFrame")
print(two_cols.head(3))

类型: <class 'pandas.DataFrame'>

取一列: <class 'pandas.Series'> → 这是一个 Series
0    110000.0
1     46000.0
2     74000.0
Name: annualIncome, dtype: float64

取多列: <class 'pandas.DataFrame'> → 还是 DataFrame
   annualIncome    dti
0      110000.0  17.05
1       46000.0  27.83
2       74000.0  22.77


In [15]:
# 常用的几个"看一眼"的方法
print("列名:", list(full.columns)[:8], "... 共", len(full.columns), "列")
print()
print("每列的类型:")
print(full.dtypes.head(8))
print()
print("只看前 3 行:")
print(full[["grade", "subGrade", "employmentLength", "issueDate"]].head(3))

列名: ['id', 'loanAmnt', 'term', 'interestRate', 'installment', 'grade', 'subGrade', 'employmentTitle'] ... 共 46 列

每列的类型:
id                   int64
loanAmnt           float64
term                 int64
interestRate       float64
installment        float64
grade                  str
subGrade               str
employmentTitle    float64
dtype: object

只看前 3 行:
  grade subGrade employmentLength   issueDate
0     E       E2          2 years  2014-07-01
1     D       D2          5 years  2012-08-01
2     D       D3          8 years  2015-10-01


## 第 2 步：`.map(字典)` —— 整列一起替换

`grade` 是 `A` 到 `G` 七个字母。它们有顺序（A 最好，G 最差），所以要变成 0 到 6。

关键：**不用写循环**。给它一个字典，它自动把这一列每个值都查一遍。

In [16]:
print("处理前:")
# 对比两种写法，第一种是转成列表好看，第二种是打印出原本的表格形式
print(full["grade"].head(5).tolist())
print(full["grade"].head(5))
print()

# 造一个字典 {"A": 0, "B": 1, ...}
grade_dict = {g: i for i, g in enumerate("ABCDEFG")}
print("这个字典长这样:", grade_dict)
print()

full["grade"] = full["grade"].map(grade_dict)# map中传一个字典

print("map 之后:")
print(full["grade"].head(5).tolist(), " ← 类型从文字变成了数字")
print(full["grade"].head(5))

处理前:
['E', 'D', 'D', 'A', 'C']
0    E
1    D
2    D
3    A
4    C
Name: grade, dtype: str

这个字典长这样: {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}

map 之后:
[4, 3, 3, 0, 2]  ← 类型从文字变成了数字
0    4
1    3
2    3
3    0
4    2
Name: grade, dtype: int64


`subGrade` 是 `A1` 到 `G5`，一共 35 档，比 `grade` 更细。用两个循环造字典：外层走 A~G，内层走 1~5。

In [17]:
print("处理前:", full["subGrade"].head(5).tolist())

subgrade_dict = {f"{g}{n}": i * 5 + n - 1
                 for i, g in enumerate("ABCDEFG") for n in range(1, 6)}

print()
print("字典的前 7 项:", dict(list(subgrade_dict.items())[:7]))
print("字典一共", len(subgrade_dict), "项")
print()

full["subGrade"] = full["subGrade"].map(subgrade_dict)

print("map 之后:", full["subGrade"].head(5).tolist())

处理前: ['E2', 'D2', 'D3', 'A4', 'C2']

字典的前 7 项: {'A1': 0, 'A2': 1, 'A3': 2, 'A4': 3, 'A5': 4, 'B1': 5, 'B2': 6}
字典一共 35 项

map 之后: [21, 16, 17, 3, 11]


## 第 3 步：处理字符串 —— `replace` + `.str.extract`

`employmentLength` 是 `"5 years"`、`"10+ years"` 这种文字。要变成数字 5、10。

分三小步：
1. `.replace({...})` 把两个特殊的写法先换掉，`"10+ years"` → `"10"`，`"< 1 year"` → `"0"`
2. `.str.extract(r"(\d+)")` 用正则把里面的数字抠出来
3. `.astype(float)` 把文字 "5" 变成数字 5.0

In [18]:
print("原始值（看看有哪些写法）:")
print(full["employmentLength"].value_counts(dropna=False))

原始值（看看有哪些写法）:
employmentLength
10+ years    32661
2 years       9176
< 1 year      8155
3 years       7856
1 year        6735
5 years       6281
4 years       5962
NaN           5830
6 years       4709
8 years       4482
7 years       4316
9 years       3837
Name: count, dtype: int64


In [19]:
s = full["employmentLength"]
print("① 原始:         ", s.head(5).tolist())

s = s.replace({"10+ years": "10", "< 1 year": "0"})
print("② replace 后:   ", s.head(5).tolist())

s = s.str.extract(r"(\d+)", expand=False)
print("③ 抠出数字后:   ", s.head(5).tolist(), " ← 注意这时候还是字符串")

s = s.astype(float)
print("④ 转成数字后:   ", s.head(5).tolist(), " ← 现在是 float")
print()
print("空值还在，符合预期:", s.isna().sum(), "个空值")
print()

full["employmentLength"] = s
print("上面四步可以写成一行（preprocess.py 里就是链式写的），效果完全一样。")

① 原始:          ['2 years', '5 years', '8 years', '10+ years', nan]
② replace 后:    ['2 years', '5 years', '8 years', '10', nan]
③ 抠出数字后:    ['2', '5', '8', '10', nan]  ← 注意这时候还是字符串
④ 转成数字后:    [2.0, 5.0, 8.0, 10.0, nan]  ← 现在是 float

空值还在，符合预期: 5830 个空值

上面四步可以写成一行（preprocess.py 里就是链式写的），效果完全一样。


## 第 4 步：日期 —— `pd.to_datetime` + `.dt`

`issueDate` 是 `"2014-07-01"` 这样的文字。`pd.to_datetime` 把它变成日期对象，之后就能用 `.dt.year`、`.dt.month` 取出年份月份。

`earliesCreditLine` 是 `"Aug-2001"` 这种格式，要告诉 pandas 怎么解析：`format="%b-%Y"` 表示"三字母月份-四位年份"。

In [22]:
print("处理前:")
print(full[["issueDate", "earliesCreditLine"]].head(3))
print()

issue = pd.to_datetime(full["issueDate"])
earliest = pd.to_datetime(full["earliesCreditLine"], format="%b-%Y")

print("转换后类型:", issue.dtype)
print("issue 前 3 个:   ", issue.head(3).tolist())
print("earliest 前 3 个:", earliest.head(3).tolist())
print()

print("用 .dt 取出年份和月份:")
print("年份:", issue.dt.year.head(3).tolist())
print("月份:", issue.dt.month.head(3).tolist())

处理前:
    issueDate earliesCreditLine
0  2014-07-01          Aug-2001
1  2012-08-01          May-2002
2  2015-10-01          May-2006

转换后类型: datetime64[us]
issue 前 3 个:    [Timestamp('2014-07-01 00:00:00'), Timestamp('2012-08-01 00:00:00'), Timestamp('2015-10-01 00:00:00')]
earliest 前 3 个: [Timestamp('2001-08-01 00:00:00'), Timestamp('2002-05-01 00:00:00'), Timestamp('2006-05-01 00:00:00')]

用 .dt 取出年份和月份:
年份: [2014, 2012, 2015]
月份: [7, 8, 10]


In [23]:
# 把日期变成数字：距离 2007 年 6 月有几个月
full["issueMonths"] = (issue.dt.year - 2007) * 12 + issue.dt.month - 6

# 信用历史有多长：放款日期 - 第一张信用卡的日期
full["creditHistoryMonths"] = ((issue.dt.year - earliest.dt.year) * 12
                               + issue.dt.month - earliest.dt.month)

print(full[["issueMonths", "creditHistoryMonths"]].head(5))
print()
print("注意：这里没有写循环，整列一起算。两个 Series 相减是逐行对应的。")

   issueMonths  creditHistoryMonths
0           85                  155
1           62                  123
2          100                  113
3           98                  195
4          105                  463

注意：这里没有写循环，整列一起算。两个 Series 相减是逐行对应的。


## 第 5 步：删掉没用的列 + 标记缺失

**删列**分两种：

- 原始列用完了就删（`issueDate`、`earliesCreditLine`）
- 本来就没什么用的列：`id` 只是个编号；`policyCode` 整列都是 1，一点信息都没有；`n3` 和 `n2` 完全一样；`ficoRangeHigh` 永远等于 `ficoRangeLow + 4`

**标记缺失**：`n0`~`n14` 大约有 5% 是空的。**"空着"本身可能就是信息**，所以额外加一列标记它是不是空。

In [11]:
print("删之前有", full.shape[1], "列")

# policyCode 整列一样，没有任何信息
print()
print("policyCode 有几种取值:", full["policyCode"].nunique(), " ← 只有一种，留着没用")
print("n2 和 n3 是不是完全一样:", (full["n2"].fillna(-9) == full["n3"].fillna(-9)).all())
print("ficoRangeHigh - ficoRangeLow 的差值分布:")
print((full["ficoRangeHigh"] - full["ficoRangeLow"]).value_counts())

删之前有 48 列

policyCode 有几种取值: 1  ← 只有一种，留着没用
n2 和 n3 是不是完全一样: True
ficoRangeHigh - ficoRangeLow 的差值分布:
4.0    99988
5.0       12
Name: count, dtype: int64


In [12]:
full = full.drop(columns=["id", "policyCode", "n3", "ficoRangeHigh",
                          "issueDate", "earliesCreditLine"])

print("删之后有", full.shape[1], "列  ← 少了 6 列")
print()
print("n0 的前 8 个值:", full["n0"].head(8).tolist())
print("n0 空值个数:", full["n0"].isna().sum())
print()
print("isna() 的样子:", full["n0"].isna().head(8).tolist(), " ← True 表示这里是空的")
print()

full["n_missing"] = full["n0"].isna().astype(float)
print("变成 0/1 之后:", full["n_missing"].head(8).tolist())
print()
print("为什么要 .astype(float)？因为神经网络只认数字，True/False 它不认。")

删之后有 42 列  ← 少了 6 列

n0 的前 8 个值: [0.0, nan, 0.0, 6.0, 1.0, 12.0, 0.0, 0.0]
n0 空值个数: 5050

isna() 的样子: [False, True, False, False, False, False, False, False]  ← True 表示这里是空的

变成 0/1 之后: [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

为什么要 .astype(float)？因为神经网络只认数字，True/False 它不认。


## 第 6 步：造新特征

已有的列相除，能得到更有用的信息。比如"贷款金额占年收入的比例"，比单独看这两个数更有意义。

**又是整列一起算，不写循环。**

In [13]:
full["loanToIncome"] = full["loanAmnt"] / (full["annualIncome"] + 1)
full["installmentToIncome"] = full["installment"] * 12 / (full["annualIncome"] + 1)

print(full[["loanAmnt", "annualIncome", "loanToIncome"]].head(5))
print()
print("第 1 行验证一下：18000 / (46000 + 1) =", round(18000 / 46001, 6))
print()
print("为什么 + 1？防止有人年收入是 0，除以 0 会变成无穷大。")

   loanAmnt  annualIncome  loanToIncome
0   35000.0      110000.0      0.318179
1   18000.0       46000.0      0.391296
2   12000.0       74000.0      0.162160
3   11000.0      118000.0      0.093220
4    3000.0       29000.0      0.103445

第 1 行验证一下：18000 / (46000 + 1) = 0.391296

为什么 + 1？防止有人年收入是 0，除以 0 会变成无穷大。


## 第 7 步：频数编码（这一步最绕）

`employmentTitle`、`title`、`postCode` 这三列，取值特别多（几十万个）。给它们做 one-hot 会一下子多出几十万列，不现实。

换个思路：**不用"是什么"，改用"出现了多少次"。** 这个值是很常见的，还是很罕见的？

`.value_counts()` 是数每个值出现了几次，`map` 把这个次数贴回去。

In [14]:
print("postCode 一共有", full["postCode"].nunique(), "种取值")
print()

vc = full["postCode"].value_counts()
print("value_counts() 的样子（每个值出现几次，从多到少排）:")
print(vc.head())
print()

print("原始的 postCode:    ", full["postCode"].head(6).tolist())
encoded = full["postCode"].map(vc)
print("map 之后:           ", encoded.head(6).tolist())
print()
first_code = full["postCode"].iloc[0]
print(f"对照第 0 行：postCode 是 {first_code}，它一共出现了 {vc[first_code]} 次")

postCode 一共有 877 种取值

value_counts() 的样子（每个值出现几次，从多到少排）:
postCode
134.0    1091
19.0     1084
51.0     1016
31.0      934
4.0       923
Name: count, dtype: int64

原始的 postCode:     [137.0, 156.0, 337.0, 148.0, 301.0, 512.0]
map 之后:            [257, 464, 210, 69, 189, 317]

对照第 0 行：postCode 是 137.0，它一共出现了 257 次


In [15]:
# 频数分布很偏（少数值出现很多次，多数只出现一两次），所以再取个 log 压一压
before = full["postCode"].map(full["postCode"].value_counts())
full["postCode"] = np.log1p(before).fillna(0)
print("取 log 之前，最大值:", before.max())
print("取 log 之后，最大值:", round(full["postCode"].max(), 3))
print()

for c in ["employmentTitle", "title"]:
    before = full[c].map(full[c].value_counts()).max()
    full[c] = np.log1p(full[c].map(full[c].value_counts())).fillna(0)
    print(f"{c:16s} 处理前最大 {before:>10,.0f}  →  处理后最大 {full[c].max():.3f}")

取 log 之前，最大值: 1091
取 log 之后，最大值: 6.996

employmentTitle  处理前最大      6,333  →  处理后最大 8.754
title            处理前最大     49,018  →  处理后最大 10.800


`np.log1p(x)` 就是 `log(1 + x)`。用 `1 + x` 是为了 x 等于 0 时也不会出错。

再看几列"长尾"的数值列，同样取 log。这些列都是大部分人很小、极少数人特别大：

In [16]:
for c in ["loanAmnt", "installment", "annualIncome", "revolBal"]:
    before = full[c].max()
    full[c] = np.log1p(full[c])
    print(f"{c:15s} 处理前最大 {before:>12,.0f}  →  处理后最大 {full[c].max():.2f}")

loanAmnt        处理前最大       40,000  →  处理后最大 10.60
installment     处理前最大        1,608  →  处理后最大 7.38
annualIncome    处理前最大   10,999,200  →  处理后最大 16.21
revolBal        处理前最大      898,146  →  处理后最大 13.71


## 第 8 步：`pd.get_dummies` —— one-hot 编码

`homeOwnership` 有 0~5 六种取值，但这个数字**不代表大小**（不是"5 比 0 更高级"），只是六个类别。

所以拆成 6 列，每列只有 0 或 1。

In [24]:
cols = ["homeOwnership", "verificationStatus", "purpose", "regionCode",
        "initialListStatus", "applicationType", "term"]

print("处理前:", full.shape[1], "列")
print(full[["homeOwnership", "purpose"]].head(3))
print()

full = pd.get_dummies(full, columns=cols, dtype=float)# ont-hot编码核心代码

print("处理后:", full.shape[1], "列  ← 变多了")
print()

onehot_cols = [c for c in full.columns if c.startswith("homeOwnership_")]
print("homeOwnership 拆成了这几列:", onehot_cols)
print(full[["homeOwnership_0", "homeOwnership_1", "homeOwnership_2"]].head(3))
print()
print("第 0 行原来的 homeOwnership 是 2，所以 homeOwnership_2 是 1，其他都是 0。")

处理前: 48 列
   homeOwnership  purpose
0              2        1
1              0        0
2              0        0

处理后: 120 列  ← 变多了

homeOwnership 拆成了这几列: ['homeOwnership_0', 'homeOwnership_1', 'homeOwnership_2', 'homeOwnership_3', 'homeOwnership_4', 'homeOwnership_5']
   homeOwnership_0  homeOwnership_1  homeOwnership_2
0              0.0              0.0              1.0
1              1.0              0.0              0.0
2              1.0              0.0              0.0

第 0 行原来的 homeOwnership 是 2，所以 homeOwnership_2 是 1，其他都是 0。


## 第 9 步：`.iloc` 把测试集切回来

刚才拼在一起了，现在要按行号切开。**`.iloc` 是按位置切**（第几行到第几行），区别于按名字切的 `.loc`。

In [25]:
X_all = full.iloc[:n_train]    # 从第 0 行到第 n_train 行（不含结尾）
X_test = full.iloc[n_train:]   # 从第 n_train 行到最后

print("X_all （训练部分）:", X_all.shape)
print("X_test（测试部分）:", X_test.shape)
print()
print("行数对得上:", len(X_all), "=", n_train)

X_all （训练部分）: (50000, 120)
X_test（测试部分）: (50000, 120)

行数对得上: 50000 = 50000


## 第 10 步：切出验证集

从训练部分里再切 20% 出来当验证集。训练时模型看不到它，用它来打分才能反映真实水平。

`stratify=y` 保证两边违约比例一样（都是 20%）。

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_all, y, test_size=0.2, random_state=42, stratify=y)

print("训练集:", X_train.shape, "违约比例", round(y_train.mean(), 4))
print("验证集:", X_val.shape, "违约比例", round(y_val.mean(), 4))
print("测试集:", X_test.shape)
print()
print("两边比例几乎一样，这就是 stratify 的作用。")

训练集: (40000, 117) 违约比例 0.2002
验证集: (10000, 117) 违约比例 0.2002
测试集: (50000, 117)

两边比例几乎一样，这就是 stratify 的作用。


## 第 11 步：统计量 —— 只能从训练集算

下面要用到"第 0.1% 分位数"、"中位数"、"均值"、"标准差"。

**这些必须只用训练集来算，再原样用到验证集和测试集上。** 如果连验证集一起算了，等于提前偷看了验证集的信息，分数会偏乐观。这个坑叫**数据泄露**。

`.quantile()`、`.median()`、`.mean()`、`.std()` 返回的都是 Series：**一列一个数**。

In [20]:
print("X_train 的形状:", X_train.shape, " ← 这是几万行的大表")
print()

lo = X_train.quantile(0.001)
print("lo = X_train.quantile(0.001) 的形状:", lo.shape)
print("← 这是一个 Series，每一列对应一个数，一共", len(lo), "个数")
print()
print("看看前 5 列的下界:")
print(lo.head())

X_train 的形状: (40000, 117)  ← 这是几万行的大表

lo = X_train.quantile(0.001) 的形状: (117,)
← 这是一个 Series，每一列对应一个数，一共 117 个数

看看前 5 列的下界:
loanAmnt        6.908755
interestRate    5.320000
installment     3.515716
grade           0.000000
subGrade        0.000000
Name: 0.001, dtype: float64


### 关键一步：大表 减 小 Series，怎么算出来的？

这是 pandas 最"魔法"的地方，也是读懂 `preprocess.py` 的最大门槛。

答案是：**按列名自动对齐**。`annualIncome` 那一列减的是年收入的中位数，`dti` 减的是 dti 的中位数。行数差多少都没关系，只要列名对得上。

In [21]:
small = X_train[["annualIncome", "dti"]].head(4)
print("4 行的大表:")
print(small)
print()

med = X_train[["annualIncome", "dti"]].median()
print("只有 2 个数的 Series（中位数）:")
print(med)
print()

print("大表 - 小 Series 的结果:")# 这么吊
print(small - med)
print()
print("验证第 0 行：", small["annualIncome"].iloc[0], "-", med["annualIncome"], "=",
      round(small["annualIncome"].iloc[0] - med["annualIncome"], 2))
print("验证第 1 行：", small["annualIncome"].iloc[1], "-", med["annualIncome"], "=",
      round(small["annualIncome"].iloc[1] - med["annualIncome"], 2))
print()
print("每一行减的都是同一个中位数；而 dti 那一列减的是 dti 的中位数。这就是'按列对齐'。")

4 行的大表:
       annualIncome    dti
28205     10.596660  23.43
3728      11.289794  24.75
35060     10.950824   4.42
4394      10.308986  24.48

只有 2 个数的 Series（中位数）:
annualIncome    11.082158
dti             17.570000
dtype: float64

大表 - 小 Series 的结果:
       annualIncome    dti
28205     -0.485498   5.86
3728       0.207636   7.18
35060     -0.131334 -13.15
4394      -0.773172   6.91

验证第 0 行： 10.596659732783579 - 11.082157933374816 = -0.49
验证第 1 行： 11.289794413577894 - 11.082157933374816 = 0.21

每一行减的都是同一个中位数；而 dti 那一列减的是 dti 的中位数。这就是'按列对齐'。


## 第 12 步：`clip` 截掉异常值

`dti` 里有个别值是 999，年收入有 0 的。这些极端值会把神经网络带偏，所以把超出范围的拉回来。

`clip(lo, hi, axis=1)` 的 `axis=1` 意思是"按列来"，因为 `lo`/`hi` 是 Series（一列一个数）。

In [22]:
lo = X_train.quantile(0.001)
hi = X_train.quantile(0.999)

demo_cols = ["annualIncome", "dti"]
demo = X_train[demo_cols].head(4).copy()

print("原始:")
print(demo)
print()
print("这两列的下界和上界（也是 Series，每列一个数）:")
print(pd.DataFrame({"lo": lo[demo_cols], "hi": hi[demo_cols]}))
print()

clipped = demo.clip(lo[demo_cols], hi[demo_cols], axis=1)
print("clip 之后:")
print(clipped)
print()
print("超出上下界的被拉到边界上，在范围内的原样不动。")

原始:
       annualIncome    dti
28205     10.596660  23.43
3728      11.289794  24.75
35060     10.950824   4.42
4394      10.308986  24.48

这两列的下界和上界（也是 Series，每列一个数）:
                   lo         hi
annualIncome  9.21042  13.176894
dti           0.14000  63.111500

clip 之后:
       annualIncome    dti
28205     10.596660  23.43
3728      11.289794  24.75
35060     10.950824   4.42
4394      10.308986  24.48

超出上下界的被拉到边界上，在范围内的原样不动。


## 第 13 步：填缺失值

`fillna` 把空的地方填上。这里填的是**训练集的中位数**（不是 0，也不是这一列自己的均值）。

为什么用中位数？因为中位数不受极端值影响，比均值更稳。

In [23]:
median = X_train.median()

print("处理前，三份数据里的空值总数:")
print("  X_train:", X_train.isna().sum().sum())
print("  X_val:  ", X_val.isna().sum().sum())
print("  X_test: ", X_test.isna().sum().sum())
print()


def clip_fill(X):
    X = X.clip(lo, hi, axis=1)   # 先截异常值
    return X.fillna(median)      # 再填空值


X_train, X_val, X_test = clip_fill(X_train), clip_fill(X_val), clip_fill(X_test)

print("处理后，空值个数:")
print("  X_train:", X_train.isna().sum().sum())
print("  X_val:  ", X_val.isna().sum().sum())
print("  X_test: ", X_test.isna().sum().sum())
print()
print("注意：lo / hi / median 都是从 X_train 算的，但用在了三个数据集上。")

处理前，三份数据里的空值总数:
  X_train: 31151
  X_val:   7857
  X_test:  39649



处理后，空值个数:
  X_train: 0
  X_val:   0
  X_test:  0

注意：lo / hi / median 都是从 X_train 算的，但用在了三个数据集上。


## 第 14 步：标准化 —— 就是视频里的 `Normalize`

各列数值范围差太多（年收入几十万，dti 只有十几），会让训练变慢、不好收敛。标准化把它们拉到同一个尺度。

`(X - mean) / std` 又用到了刚才那个"大表和小 Series"的对齐规则。

In [24]:
mean, std = X_train.mean(), X_train.std().replace(0, 1)
print("为什么要 .replace(0, 1)？因为万一有某列标准差是 0（整列一样的值），除以 0 会出错。")
print()

demo_cols = ["annualIncome", "dti", "loanAmnt"]
print("标准化前的几列:")
print(X_train[demo_cols].head(3).round(1))
print()
print("这几列的均值和标准差（都只是 Series）:")
print(pd.DataFrame({"mean": mean[demo_cols].round(2), "std": std[demo_cols].round(2)}))
print()

X_train = ((X_train - mean) / std).to_numpy(np.float32)
X_val = ((X_val - mean) / std).to_numpy(np.float32)
X_test = ((X_test - mean) / std).to_numpy(np.float32)

print("处理后的前 3 行、前 3 列:")
print(X_train[:3, :3].round(3))
print()
print("现在都是 0 附近的小数字了。而且注意：")
print("类型变成了", type(X_train), "，dtype 是", X_train.dtype, "，形状", X_train.shape)
print("← 已经不是 DataFrame 了，是 numpy 数组，可以直接喂给 PyTorch。")

为什么要 .replace(0, 1)？因为万一有某列标准差是 0（整列一样的值），除以 0 会出错。

标准化前的几列:
       annualIncome   dti  loanAmnt
28205          10.6  23.4       9.4
3728           11.3  24.8       8.9
35060          11.0   4.4       9.2

这几列的均值和标准差（都只是 Series）:
               mean   std
annualIncome  11.08  0.53
dti           18.18  8.64
loanAmnt       9.37  0.70



处理后的前 3 行、前 3 列:
[[ 0.035  0.582 -0.329]
 [-0.665 -0.648 -0.636]
 [-0.227 -1.26  -0.238]]

现在都是 0 附近的小数字了。而且注意：
类型变成了 <class 'numpy.ndarray'> ，dtype 是 float32 ，形状 (40000, 117)
← 已经不是 DataFrame 了，是 numpy 数组，可以直接喂给 PyTorch。


In [25]:
print("最后检查一遍，训练集所有特征的均值和标准差:")
print("  均值   :", round(float(X_train.mean()), 4), " ← 应该接近 0")
print("  标准差 :", round(float(X_train.std()), 4), " ← 应该接近 1")
print()
print("特征个数:", X_train.shape[1])
print("preprocess.py 用全部 80 万行跑出来是 118 个，这里少了 1 个。")
print()
print("少的正是 regionCode_50：这个地区只出现在数据的后半段，前 5 万行里一个都没有。")
print()
print("这正好说明了 preprocess.py 为什么先 concat 再 get_dummies：")
print("如果 train 和 test 分开做 one-hot，一边有 regionCode_50 另一边没有，")
print("列就对不齐了，模型也没法用。")

最后检查一遍，训练集所有特征的均值和标准差:
  均值   : -0.0  ← 应该接近 0
  标准差 : 0.9784  ← 应该接近 1

特征个数: 117
preprocess.py 用全部 80 万行跑出来是 118 个，这里少了 1 个。

少的正是 regionCode_50：这个地区只出现在数据的后半段，前 5 万行里一个都没有。

这正好说明了 preprocess.py 为什么先 concat 再 get_dummies：
如果 train 和 test 分开做 one-hot，一边有 regionCode_50 另一边没有，
列就对不齐了，模型也没法用。


## 第 15 步：存成文件

numpy 数组用 `np.savez_compressed` 存成一个 `.npz` 文件，之后 `dataset.py` 直接读它。

In [26]:
import os
os.makedirs("data", exist_ok=True)

np.savez_compressed("data/demo_processed.npz",
                    X_train=X_train, y_train=y_train,
                    X_val=X_val, y_val=y_val,
                    X_test=X_test)

print("存好了: data/demo_processed.npz")
print()
print("读回来看看:")
d = np.load("data/demo_processed.npz")
for k in d.files:
    print("  ", k, d[k].shape)
print()
print("注意：这只是一个练习用的文件，别和 preprocess.py 生成的 data/processed.npz 搞混。")

存好了: data/demo_processed.npz

读回来看看:
   X_train (40000, 117)
   y_train (40000,)
   X_val (10000, 117)
   y_val (10000,)
   X_test (50000, 117)

注意：这只是一个练习用的文件，别和 preprocess.py 生成的 data/processed.npz 搞混。


---

# 回头再读 preprocess.py

现在从上到下再看一遍 [preprocess.py](preprocess.py)，每一行应该都能对应到这个 notebook 里的某一格。

## 三个最容易忘的思维

1. **不写循环。** 整个文件没有一处 `for` 遍历每一行，都是整列一起算。
2. **Series 自动对齐。** 大表减小 Series，是按列名一一对应的。
3. **链式调用从里往外读。** 比如 `df["employmentLength"].replace(...).str.extract(...).astype(float)` 就是第 3 步的 ①②③④，拆成四行写完全一样。

## 练习题（自己在这个 notebook 里加格子做）

1. `n0` 到 `n14` 这些列，哪些空值最多？用 `.isna().mean()` 排个序看看。
2. 算一下 `interestRate` 和 `isDefault` 的关系：按利率分成 5 组，看看每组的违约率是不是递增的？（提示：`pd.cut` + `groupby`；注意 `y` 是 numpy 数组，可以先拼回一个 DataFrame）
3. 现在 `dti` 的 0.1% 分位数和 99.9% 分位数分别是多少？大概有多少行被 clip 改动了？
4. 把 `grade` 改成用 `pd.get_dummies` 做 one-hot（而不是 0~6 映射），列数会多多少？想想哪种更合适，为什么。

In [29]:
print(round(full[[f'n{i}' for i in range(15)]].mean().mean()), 4)

5 4


In [30]:
print(round(full[[f'n{i}' for i in range(15)]].mean().mean(), 4))

5.2835


In [32]:
cols = ['n0', 'n1', 'n2', 'n3', 'n4', 'n5', 'n6', 'n7', 'n8', 'n9', 'n10', 'n11', 'n12', 'n13', 'n14']
print(full[cols].mean().round(4))


n0      0.5061
n1      3.6448
n2      5.6473
n3      5.6473
n4      4.7341
n5      8.0868
n6      8.5667
n7      8.2900
n8     14.6103
n9      5.5955
n10    11.6509
n11     0.0010
n12     0.0036
n13     0.0916
n14     2.1770
dtype: float64


In [34]:
print(full[cols].isna().astype(float))

        n0   n1   n2   n3   n4   n5   n6   n7   n8   n9  n10  n11  n12  n13  n14
0      0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
1      1.0  1.0  1.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0  0.0  1.0  1.0  1.0  1.0
2      0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
3      0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
4      0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
...    ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...
99995  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0
99996  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
99997  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
99998  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
99999  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0

[100000 rows x 15 columns]


In [35]:
temp = full[cols].isna().astype(float)

In [36]:
print(temp.mean())

n0     0.05050
n1     0.05050
n2     0.05050
n3     0.05050
n4     0.04204
n5     0.05050
n6     0.05050
n7     0.05050
n8     0.05050
n9     0.05050
n10    0.04204
n11    0.08729
n12    0.05050
n13    0.05050
n14    0.05050
dtype: float64
